In [1]:
import pandas as pd
import numpy as np
import os
from scipy.spatial.distance import cdist
import seaborn as sns

In [2]:
def compute_neighbors_with_index(df, metric='sqeuclidean'):
    """
    Computes pairwise distances and returns a dictionary where values are
    neighbor indices sorted from closest to farthest.
    """
    data_array = df.to_numpy()
    indices = df.index.tolist()
    
    # 1. Compute pairwise distance matrix
    # Resulting shape: (N, N)
    dist_matrix = cdist(data_array, data_array, metric=metric)
    
    # 2. Get indices that sort distances ascending (closest first)
    # The index at position 0 will always be the point itself.
    sorted_idx_positions = np.argsort(dist_matrix, axis=1)
    
    # 3. Map numerical positions back to the original DataFrame index labels
    neighbors_dict = {}
    for i, idx_label in enumerate(indices):
        neighbors_dict[idx_label] = [indices[pos] for pos in sorted_idx_positions[i]]
        
    return neighbors_dict

In [3]:
def compute_jaccard_similarity(nbors_dict_1, nbors_dict_2, k):
    """
    Computes the average Jaccard similarity for the top-k neighbors.
    Note: indices[0] is the point itself, so we slice [1:k+1].
    """
    jaccard_scores = []
    
    # Iterate through all samples present in both dictionaries
    for sample_id in nbors_dict_1.keys():
        # Get top-k neighbors (excluding self)
        set_1 = set(nbors_dict_1[sample_id][1:k+1])
        set_2 = set(nbors_dict_2[sample_id][1:k+1])
        
        intersection = len(set_1.intersection(set_2))
        union = len(set_1.union(set_2))
        
        # Avoid division by zero if sets are somehow empty
        score = intersection / union if union > 0 else 0
        jaccard_scores.append(score)
        
    return np.mean(jaccard_scores)

In [4]:
base_path = "../../data/embeddings/"
methods = ["UMAP", "PCA", "tSNE"]
datasets = ["HANCOCK", "MIMIC", "TCGA_LUAD"]
k_list = [10, 30, 50]
dims = [16, 32, 64]
NUM_RUNS = 10
result_dict = {"run": [], "dim": [], "method": [], "K": [], "jaccard_index": [], "dataset": []}

In [ ]:
for data in datasets:
    print("Processing dataset = ", data)
    for dim in dims:
        print("Processing dimension = ", dim)
        for run in range(NUM_RUNS):
            vis_path = os.path.join(base_path, data, "visualization", f"{data}_samples_{dim}_{run}.tsv")
            vis_df = pd.read_csv(vis_path, sep='\t', index_col=0)
            vis_df.index = vis_df.index.astype("str")
            embedding_path = os.path.join(base_path, data, "embeddings", f"{data}_samples_{dim}_{run}.tsv")
            embedding_df = pd.read_csv(embedding_path, sep='\t', index_col=0)
            embedding_df.index = embedding_df.index.astype("str")
            embedding_nbors = compute_neighbors_with_index(embedding_df)
            for method in methods:
                method_df = vis_df[[f"{method}_1", f"{method}_2"]]
                method_nbors = compute_neighbors_with_index(method_df)
                for k in k_list:
                    avg_jaccard = compute_jaccard_similarity(embedding_nbors, method_nbors, k)
            
                    result_dict["run"].append(run)
                    result_dict["dim"].append(dim)
                    result_dict["method"].append(method)
                    result_dict["K"].append(k)
                    result_dict["jaccard_index"].append(avg_jaccard)
                    result_dict["dataset"].append(data)

result_df = pd.DataFrame(result_dict)

In [ ]:
result_df.to_csv("visualization_distance_preservation_results.csv", index=False)